In [18]:
import pandas as pd
import numpy as np

In [19]:
Hd = pd.read_csv("KaggleV2-May-2016.csv")
Hd.columns

Index(['PatientId', 'AppointmentID', 'Gender', 'ScheduledDay',
       'AppointmentDay', 'Age', 'Neighbourhood', 'Scholarship', 'Hipertension',
       'Diabetes', 'Alcoholism', 'Handcap', 'SMS_received', 'No-show'],
      dtype='str')

In [20]:
# Renaming Columns for consisency
Hd = Hd.rename(columns={
    'Hipertension':'Hypertension',
    'Handcap':'Handicap',
    'No-show': 'No_Show'
})

In [21]:
# Fix data types
# converting patientID to integer
Hd['PatientId'] = Hd['PatientId'].astype(int)

# converting cheduledDay and AppointedDay to datetime
Hd['ScheduledDay'] = pd.to_datetime(Hd['ScheduledDay']).dt.normalize()
Hd['AppointmentDay'] = pd.to_datetime(Hd['AppointmentDay']).dt.normalize()


In [22]:
# Cleaning the age column
Hd = Hd.drop(Hd[Hd['Age'] < 0].index)


# Check
print('Negative ages:', (Hd['Age'] < 0).sum())
print("Age > 100:",(Hd['Age'] > 100).sum())


Negative ages: 0
Age > 100: 7


In [23]:
Hd.head(10)

,PatientId,AppointmentID,Gender,ScheduledDay,AppointmentDay,Age,Neighbourhood,Scholarship,Hypertension,Diabetes,Alcoholism,Handicap,SMS_received,No_Show
0,29872499824296,5642903,F,2016-04-29 00:00:00+00:00,2016-04-29 00:00:00+00:00,62,JARDIM DA PENHA,0,1,0,0,0,0,No
1,558997776694438,5642503,M,2016-04-29 00:00:00+00:00,2016-04-29 00:00:00+00:00,56,JARDIM DA PENHA,0,0,0,0,0,0,No
2,4262962299951,5642549,F,2016-04-29 00:00:00+00:00,2016-04-29 00:00:00+00:00,62,MATA DA PRAIA,0,0,0,0,0,0,No
3,867951213174,5642828,F,2016-04-29 00:00:00+00:00,2016-04-29 00:00:00+00:00,8,PONTAL DE CAMBURI,0,0,0,0,0,0,No
4,8841186448183,5642494,F,2016-04-29 00:00:00+00:00,2016-04-29 00:00:00+00:00,56,JARDIM DA PENHA,0,1,1,0,0,0,No
5,95985133231274,5626772,F,2016-04-27 00:00:00+00:00,2016-04-29 00:00:00+00:00,76,REPÚBLICA,0,1,0,0,0,0,No
6,733688164476661,5630279,F,2016-04-27 00:00:00+00:00,2016-04-29 00:00:00+00:00,23,GOIABEIRAS,0,0,0,0,0,0,Yes
7,3449833394123,5630575,F,2016-04-27 00:00:00+00:00,2016-04-29 00:00:00+00:00,39,GOIABEIRAS,0,0,0,0,0,0,Yes
8,56394729949972,5638447,F,2016-04-29 00:00:00+00:00,2016-04-29 00:00:00+00:00,21,ANDORINHAS,0,0,0,0,0,0,No
9,78124564369297,5629123,F,2016-04-27 00:00:00+00:00,2016-04-29 00:00:00+00:00,19,CONQUISTA,0,0,0,0,0,0,No


In [24]:
# Creating new useful columns
# creating new columns from ApppointmentDay and ScheduleDy
Hd['Waiting_Days'] = (Hd['AppointmentDay'] - Hd['ScheduledDay']).dt.days

Hd['Appointment_Weekday'] = Hd['AppointmentDay'].dt.day_name()

Hd['Scheduled_Weekday'] = Hd['ScheduledDay'].dt.day_name()

# Creating new categories from the waiting

def get_group(time):
    if time < 0:
        return 'Came  earlier'
    elif time ==0:
        return 'Same Day'
    elif 0 < time <=7:
        return "1-7days"
    elif time <= 30:
        return 'Within 30days'
    else:
        return 'More than 30days'
Hd['Timing'] = Hd['Waiting_Days'].apply(get_group)
#creating new column called Age_Group
Hd['Age_Group'] = pd.cut(Hd['Age'],
                        bins=[-1,17,59,120],
                        labels=['Child', 'Adult', "Elderly"])


In [25]:
Hd[['PatientId', 'AppointmentID', 'Gender','Age', 'Neighbourhood', 'No_Show', 'Waiting_Days', 'Appointment_Weekday', 'Scheduled_Weekday', 'Age_Group', 'Timing']].head(15)

,PatientId,AppointmentID,Gender,Age,Neighbourhood,No_Show,Waiting_Days,Appointment_Weekday,Scheduled_Weekday,Age_Group,Timing
0,29872499824296,5642903,F,62,JARDIM DA PENHA,No,0,Friday,Friday,Elderly,Same Day
1,558997776694438,5642503,M,56,JARDIM DA PENHA,No,0,Friday,Friday,Adult,Same Day
2,4262962299951,5642549,F,62,MATA DA PRAIA,No,0,Friday,Friday,Elderly,Same Day
3,867951213174,5642828,F,8,PONTAL DE CAMBURI,No,0,Friday,Friday,Child,Same Day
4,8841186448183,5642494,F,56,JARDIM DA PENHA,No,0,Friday,Friday,Adult,Same Day
5,95985133231274,5626772,F,76,REPÚBLICA,No,2,Friday,Wednesday,Elderly,1-7days
6,733688164476661,5630279,F,23,GOIABEIRAS,Yes,2,Friday,Wednesday,Adult,1-7days
7,3449833394123,5630575,F,39,GOIABEIRAS,Yes,2,Friday,Wednesday,Adult,1-7days
8,56394729949972,5638447,F,21,ANDORINHAS,No,0,Friday,Friday,Adult,Same Day
9,78124564369297,5629123,F,19,CONQUISTA,No,2,Friday,Wednesday,Adult,1-7days


In [26]:
(Hd['Waiting_Days'] < 0).sum()

np.int64(5)

In [27]:
Hd[Hd['Waiting_Days'] < 0]

,PatientId,AppointmentID,Gender,ScheduledDay,AppointmentDay,Age,Neighbourhood,Scholarship,Hypertension,Diabetes,Alcoholism,Handicap,SMS_received,No_Show,Waiting_Days,Appointment_Weekday,Scheduled_Weekday,Timing,Age_Group
27033,7839272661752,5679978,M,2016-05-10 00:00:00+00:00,2016-05-09 00:00:00+00:00,38,RESISTÊNCIA,0,0,0,0,1,0,Yes,-1,Monday,Tuesday,Came earlier,Adult
55226,7896293967868,5715660,F,2016-05-18 00:00:00+00:00,2016-05-17 00:00:00+00:00,19,SANTO ANTÔNIO,0,0,0,0,1,0,Yes,-1,Tuesday,Wednesday,Came earlier,Adult
64175,24252258389979,5664962,F,2016-05-05 00:00:00+00:00,2016-05-04 00:00:00+00:00,22,CONSOLAÇÃO,0,0,0,0,0,0,Yes,-1,Wednesday,Thursday,Came earlier,Adult
71533,998231581612122,5686628,F,2016-05-11 00:00:00+00:00,2016-05-05 00:00:00+00:00,81,SANTO ANTÔNIO,0,0,0,0,0,0,Yes,-6,Thursday,Wednesday,Came earlier,Elderly
72362,3787481966821,5655637,M,2016-05-04 00:00:00+00:00,2016-05-03 00:00:00+00:00,7,TABUAZEIRO,0,0,0,0,0,0,Yes,-1,Tuesday,Wednesday,Came earlier,Child


In [28]:
Hd_original = Hd[Hd['Waiting_Days'] >= 0].copy()

In [29]:
Hd_original[Hd_original['Waiting_Days'] < 0]

,PatientId,AppointmentID,Gender,ScheduledDay,AppointmentDay,Age,Neighbourhood,Scholarship,Hypertension,Diabetes,Alcoholism,Handicap,SMS_received,No_Show,Waiting_Days,Appointment_Weekday,Scheduled_Weekday,Timing,Age_Group


In [30]:
print(Hd_original.shape)
print(Hd_original.isnull().sum())
print(Hd_original['Timing'].value_counts())
print(Hd_original['Age_Group'].value_counts())

(110521, 19)
PatientId              0
AppointmentID          0
Gender                 0
ScheduledDay           0
AppointmentDay         0
Age                    0
Neighbourhood          0
Scholarship            0
Hypertension           0
Diabetes               0
Alcoholism             0
Handicap               0
SMS_received           0
No_Show                0
Waiting_Days           0
Appointment_Weekday    0
Scheduled_Weekday      0
Timing                 0
Age_Group              0
dtype: int64
Timing
Same Day            38562
1-7days             32185
Within 30days       29396
More than 30days    10378
Name: count, dtype: int64
Age_Group
Adult      61971
Child      27378
Elderly    21172
Name: count, dtype: int64


In [31]:
print('final shape:', Hd_original.shape)
print(Hd_original.isnull().sum())
print(Hd_original['No_Show'].value_counts(normalize=True).round(3))
print(Hd_original['Age_Group'].value_counts())

final shape: (110521, 19)
PatientId              0
AppointmentID          0
Gender                 0
ScheduledDay           0
AppointmentDay         0
Age                    0
Neighbourhood          0
Scholarship            0
Hypertension           0
Diabetes               0
Alcoholism             0
Handicap               0
SMS_received           0
No_Show                0
Waiting_Days           0
Appointment_Weekday    0
Scheduled_Weekday      0
Timing                 0
Age_Group              0
dtype: int64
No_Show
No     0.798
Yes    0.202
Name: proportion, dtype: float64
Age_Group
Adult      61971
Child      27378
Elderly    21172
Name: count, dtype: int64


In [32]:
print(Hd.shape)
print(Hd_original.shape)

(110526, 19)
(110521, 19)


In [ ]:
Hd_original.to_csv('appointments_cleansed.csv', index= False)

: 